In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
def read_and_info(filepath):
    """Load a Stata file, print path + shape + first rows (as in BA Q1)."""
    df = pd.read_stata(filepath, convert_categoricals=False)
    print(f"{filepath}: {df.shape[0]} rows, {df.shape[1]} columns")
    display(df.head().iloc[:, :8])
    return df

main   = read_and_info("../data/raw/coding_suseti_pmt.dta")
labels = read_and_info("../data/raw/mistargeting_CORRECTED.dta")

# 1. Check the merge keys before merging (BA Q5.1)
print("hhid unique in main:          ", main["hhid"].is_unique) #hhid is the household id - which is in both datasets -> to merge this must first ascertain that its unique
print("hhid_suseti unique in labels: ", labels["hhid_suseti"].is_unique)
assert main["hhid"].is_unique and labels["hhid_suseti"].is_unique


../data/raw/coding_suseti_pmt.dta: 5756 rows, 347 columns


,hhid,hhmale,hhage,hhmarried,hhage2,hhmalemarr,hheduc1,hheduc2
0,001025,1,80.0,1,6400.0,1,1,0
1,001302,1,58.0,1,3364.0,1,0,0
2,001468,1,35.0,1,1225.0,1,0,0
3,001473,1,68.0,1,4624.0,1,0,1
4,001476,0,78.0,0,6084.0,0,1,0


../data/raw/mistargeting_CORRECTED.dta: 5756 rows, 95 columns


,hhea,avg_CONSUMPTION,avg_EDUCATIONHHHEAD,avg_PMTSCORE,SHAREAGRICULTURALHH,avg_YREDUCATIONHHHEAD,EDUCATIONRTHEAD,YREDUCATIONRTHEAD
0,1,526.538879,2.777778,12.664343,100.0,6.333333,3,6
1,1,526.538879,2.777778,12.664343,100.0,6.333333,3,6
2,1,526.538879,2.777778,12.664343,100.0,6.333333,3,6
3,1,526.538879,2.777778,12.664343,100.0,6.333333,3,6
4,1,526.538879,2.777778,12.664343,100.0,6.333333,3,6


hhid unique in main:           True
hhid_suseti unique in labels:  True


In [ ]:
# 2. Reduce both datasets to the columns we need.
# We list what we KEEP (not what we drop): any column not listed is removed,
# so no leakage column can slip through by being forgotten in a drop list.

# --- Columns that are not features: IDs, grouping, quota, baseline, targets, filter helpers
id_cols       = ["hhid", "hhea", "kabu", "prop"]   # household ID, village, district, province
quota_cols    = ["share"]                           # village quota for the selection rule
baseline_cols = ["pcexppred"]   #expected consumption by government - government formula score (baseline only, never a feature)
target_cols   = ["LOGCONSUMPTION"]                  # regression target (poor comes from labels)
helper_cols   = ["age1564",                         # only to build no_workingage, dropped after
                 "hheduc1"]                         # only to find heads with missing education (all 4 hheduc = 0), dropped after

# --- Set A: the 62 variables of the government PMT formula (those with a c_* coefficient)
set_a_dwelling  = ["pcfloor", "tfloor", "twall", "troof", "toilet", "water", "lighting", "fcook", "house"]
set_a_household = ["hhsize", "hhsize2", "age04", "eschild", "jschild", "sschild", "depratio"]
set_a_head      = ["hhage", "hhage2", "hhmale", "hhmarried", "hhmalemarr",
                   "hheduc2", "hheduc3", "hheduc4", "hhsector1", "hhsector2", "hhsector3",
                   "formal", "informal"]
set_a_educ      = ["higheduc2", "higheduc3", "higheduc4"]
set_a_credit    = ["credit"]                        # self-reported, but in the government formula
set_a_location  = ["urban"]
set_a_village   = ["vill2dist", "villsd", "villsmp", "villpusk", "villpolin", "villposy",
                   "villdokt", "villbidan", "villrtype", "villpmplace", "villcred"]
set_a_assets    = ["se2_17", "se2_11_12", "se2_125", "se2_126_127", "se2_119", "se2_117",
                   "se2_115", "se2_13_15", "se2_14", "se2_114", "se2_18", "se2_124",
                    "se2_122_123", "se5_110", "se5_11_12",
                   "se5_15_16", "se2_138"]          # goat/sheep and agricultural land are in the formula too
set_a = (set_a_dwelling + set_a_household + set_a_head + set_a_educ + set_a_credit
         + set_a_location + set_a_village + set_a_assets)

# --- Set B: extra items an enumerator can see or verify
set_b_assets = ["se2_16", "se2_111", "se2_110", "se2_118", "se2_112", "se2_121",
                "se2_132_133", "se2_128", "se2_129", "se5_17", "se5_13", "se2_139",
                "FLOORTOPQUALITY"]
set_b_counts = [c for c in main.columns if c.startswith("hr06")]   # number of each asset owned
set_b = set_b_assets + set_b_counts

main_cols   = id_cols + quota_cols + baseline_cols + target_cols + helper_cols + set_a + set_b
labels_cols = ["hhid_suseti", "poor", "povline_poor", "RTHEAD"]   # key, target, its line, filter

# Checks on the lists themselves
assert len(set_a) == 62, f"set A should have 62 variables, has {len(set_a)}"
assert len(main_cols) == len(set(main_cols)), "a column is listed twice"
missing = [c for c in main_cols if c not in main.columns] + [c for c in labels_cols if c not in labels.columns]
assert not missing, f"columns not found: {missing}"

# Reduce: everything not listed above is dropped
print("before:", main.shape, labels.shape)
main   = main[main_cols].copy()
labels = labels[labels_cols].copy()
print("after: ", main.shape, labels.shape)

# 3. Leakage check: nothing derived from consumption, the government score,
# the allocation or self-reports survived the reduction
# consumption related varianbles contain the answer to what we want to predict so drop it
forbidden_prefixes = ("c_", "comp", "temp", "rank", "nhhrank", "RTS", "MISTARGET", "PMT", "RANK", "avg_")
forbidden_cols = ["CONSUMPTION", "hhconsumption", "wfood", "mfood", "mnonfood", "lnpcexpe", "epcexpe",
                  "povertygap", "verypoor", "healthcare", "shock", "maintreatment", "quota", "quota_final"]
leaks = [c for c in list(main.columns) + list(labels.columns)
         if c.startswith(forbidden_prefixes) or c in forbidden_cols or c.endswith(("_avg", "_dm"))]
assert not leaks, f"leakage columns still present: {leaks}"
print("leakage check passed")

In [ ]:
# 4. Give every column the data type that matches what it represents (as in BA Q2).
#    Types: text -> "string", whole numbers -> "Int64" (nullable: keeps missing values as <NA>),
#    measurements with decimals -> "float64".
#    Many columns were read as int8 (max 127) or float32 (less precise): converting them to
#    Int64 / float64 avoids silent overflow later (e.g. 12**2 = 144 does not fit in int8).

# --- 1. Before: how pandas read the columns
print("before:", main.dtypes.astype(str).value_counts().to_dict(), "|", labels.dtypes.astype(str).to_dict())

# --- 2. Schema: which type each group of columns should have
id_text   = ["hhid", "hhid_suseti"]                 # identifiers: labels, not numbers (keep leading zeros "001025")
id_codes  = ["hhea", "kabu", "prop"]                # village / district / province codes: whole numbers, not measurements
measures  = ["share", "pcexppred", "LOGCONSUMPTION", "pcfloor", "depratio",
             "vill2dist", "povline_poor"]           # measurements with decimals
counts    = ["hhsize", "hhsize2", "age04", "eschild", "jschild", "sschild",
             "age1564", "hhage", "hhage2"]          # whole-number counts (age in years, age squared)
hr06_whole   = [c for c in set_b_counts if c != "hr0627"]   # number of each asset owned: whole numbers
hr06_decimal = ["hr0627"]                                   # jewelry: has values like 0.5 (likely grams) -> keep decimals
binary    = [c for c in set_a + set_b_assets + ["hheduc1"]
             if c not in measures + counts]         # 0/1 dummies (se2_139 is still coded 1/3; fixed in the cleaning step)
labels_codes = ["poor", "RTHEAD"]                   # 0/1 target and filter flag

desired_dtypes = {}
desired_dtypes.update({c: "string"  for c in id_text})
desired_dtypes.update({c: "Int64"   for c in id_codes + counts + hr06_whole + binary + labels_codes})
desired_dtypes.update({c: "float64" for c in measures + hr06_decimal})

# Every kept column must have exactly one type
all_cols = list(main.columns) + list(labels.columns)
assert set(desired_dtypes) == set(all_cols), \
    f"no type for: {set(all_cols) - set(desired_dtypes)} | not in data: {set(desired_dtypes) - set(all_cols)}"

# --- 3. Apply the schema with one loop over both datasets
for df in (main, labels):
    for col, t in desired_dtypes.items():
        if col in df.columns:
            df[col] = df[col].astype(t)

# --- 4. After: verify
print("after: ", main.dtypes.astype(str).value_counts().to_dict(), "|", labels.dtypes.astype(str).to_dict())
print("hhid examples (leading zeros kept):", main["hhid"].head(3).tolist())
assert (main["hhsize"] ** 2 == main["hhsize2"]).all(), "hhsize2 should equal hhsize squared"   # failed with int8

In [ ]:
# 5. Diagnose (BA Q3.1): look for problems in each dataset BEFORE fixing anything.
#    Nothing is changed in this cell; it only prints what the cleaning step has to deal with.

# --- 1. Missing values per column (only columns with at least one missing value)
for name, df in [("main", main), ("labels", labels)]:
    na = df.isna().sum()
    na = na[na > 0].sort_values(ascending=False)
    print(f"\n{name} {df.shape}: {len(na)} columns with missing values")
    print(na.to_string())

# --- 2. Duplicated identifiers and households per village
print("\nduplicated hhid in main:          ", main["hhid"].duplicated().sum())
print("duplicated hhid_suseti in labels: ", labels["hhid_suseti"].duplicated().sum())
print("households per village:           ", main.groupby("hhea").size().value_counts().to_dict())
print("subvillage heads per village:     ", labels.merge(main[["hhid", "hhea"]], left_on="hhid_suseti", right_on="hhid")
                                                 .groupby("hhea")["RTHEAD"].sum().value_counts().to_dict())

# --- 3. Implausible values: things that cannot be true or contradict each other
binary_cols = binary + ["poor", "RTHEAD"]
checks = {
    # ranges
    "hhsize < 1":                           (main["hhsize"] < 1).sum(),
    "hhage < 15 or > 100":                  ((main["hhage"] < 15) | (main["hhage"] > 100)).sum(),
    "pcfloor <= 0":                         (main["pcfloor"] <= 0).sum(),
    "pcfloor > 200 m² per person":          (main["pcfloor"] > 200).sum(),
    "depratio < 0":                         (main["depratio"] < 0).sum(),
    "vill2dist <= 0":                       (main["vill2dist"] <= 0).sum(),
    "share outside (0, 1]":                 ((main["share"] <= 0) | (main["share"] > 1)).sum(),
    "pcexppred < 0 (negative predicted consumption)": (main["pcexppred"] < 0).sum(),
    "asset count (hr06xx) <= 0":            int((main[set_b_counts] <= 0).sum().sum()),
    # coding
    "0/1 columns with other values":        [c for c in binary_cols
                                             if not set((main if c in main else labels)[c].dropna().unique()) <= {0, 1}],
    # consistency between columns
    "hhsize2 != hhsize²":                   (main["hhsize2"] != main["hhsize"] ** 2).sum(),
    "hhage2 != hhage²":                     (main["hhage2"] != main["hhage"] ** 2).sum(),
    "age04 > hhsize":                       (main["age04"] > main["hhsize"]).sum(),
    "children in school > hhsize":          ((main["eschild"] + main["jschild"] + main["sschild"]) > main["hhsize"]).sum(),
    "head education missing (all 4 hheduc = 0)": (main[["hheduc1", "hheduc2", "hheduc3", "hheduc4"]].sum(axis=1) == 0).sum(),
    "head education: more than one hheduc = 1":  (main[["hheduc1", "hheduc2", "hheduc3", "hheduc4"]].sum(axis=1) > 1).sum(),
    "depratio missing & age1564 == 0":      (main["depratio"].isna() & (main["age1564"] == 0)).sum(),
}
print("\nimplausible / inconsistent values:")
for k, v in checks.items():
    print(f"  {k:48s} {v}")

# --- 4. Asset counts (hr06xx) vs. the matching 0/1 asset dummy
#     Expected: count missing <=> dummy == 0 (household does not own the item)
count_to_dummy = {"hr0609": "se2_17", "hr0610": "se2_16", "hr0611": "se2_111", "hr0612": "se2_110",
                  "hr0613": "se2_115", "hr0614": "se2_13_15", "hr0615": "se2_11_12", "hr0616": "se2_14",
                  "hr0617": "se2_114", "hr0618": "se2_117", "hr0619": "se2_119", "hr0620": "se2_118",
                  "hr0621": "se2_124", "hr0622": "se2_128", "hr0623": "se2_125", "hr0625": "se2_126_127",
                  "hr0626": "se2_129", "hr0627": "se2_122_123", "hr0628": "se5_110", "hr0629": "se5_17",
                  "hr0630": "se5_15_16", "hr0631": "se5_11_12", "hr0632": "se5_13", "hr0633": "se2_132_133",
                  "hr0634": "se2_112", "hr0635": "se2_121"}      # hr0607, hr0608: unlabelled, no dummy
rows = []
for cnt, dum in count_to_dummy.items():
    rows.append({"count": cnt, "dummy": dum,
                 "count missing, dummy=0 (ok: none owned)":  (main[cnt].isna() & (main[dum] == 0)).sum(),
                 "count missing, dummy=1 (problem)":         (main[cnt].isna() & (main[dum] == 1)).sum(),
                 "count present, dummy=0 (contradiction)":   (main[cnt].notna() & (main[dum] == 0)).sum()})
hr = pd.DataFrame(rows).set_index("count")
print("\nasset counts vs. dummies (only rows with a problem):")
print(hr[(hr.iloc[:, 2] > 0) | (hr.iloc[:, 3] > 0)].to_string())

# --- 5. Targets
print("\nLOGCONSUMPTION range:", main["LOGCONSUMPTION"].min().round(2), "to", main["LOGCONSUMPTION"].max().round(2))
q1, q3 = main["LOGCONSUMPTION"].quantile([0.25, 0.75])
print("Tukey outliers on log consumption (k=3):",
      ((main["LOGCONSUMPTION"] < q1 - 3 * (q3 - q1)) | (main["LOGCONSUMPTION"] > q3 + 3 * (q3 - q1))).sum())
chk = main[["hhid", "LOGCONSUMPTION"]].merge(labels, left_on="hhid", right_on="hhid_suseti")
chk = chk[chk["poor"].notna()]
on_line = np.isclose(np.exp(chk["LOGCONSUMPTION"]), chk["povline_poor"], rtol=1e-6)   # consumption exactly = poverty line
below   = (np.exp(chk["LOGCONSUMPTION"]) < chk["povline_poor"]) & ~on_line
print("poor share:", round(chk["poor"].mean(), 3))
print("poor != (consumption below the line), away from the line:", ((below.astype(int) != chk["poor"]) & ~on_line).sum())
print("households exactly on the poverty line:", on_line.sum(),
      "| of these coded poor:", int(chk.loc[on_line, "poor"].sum()))

### Diagnosis: what we found (all 5,756 households, before any row is dropped)

<details>
<summary><b>Click to open: findings and what the cleaning step does about each</b></summary>

**No problem (checked and fine)**
- No duplicated IDs in either file; both merge keys are unique.
- Exactly **one subvillage head per village** (640 in total).
- `hhsize2 = hhsize²` and `hhage2 = hhage²` everywhere (after the type fix; with `int8` this check failed).
- No household has more young children or school children than members; no negative or zero sizes, distances or floor areas; `share` is between 0 and 1.
- `poor` matches "consumption below the poverty line" for every household away from the line.

**Things the cleaning step must handle**

| Finding | Count | Decision |
|---|---|---|
| Villages with only 8 instead of 9 households | 4 | Keep. Check "≤ 9 per village" instead of "exactly 9". |
| `LOGCONSUMPTION` and `poor` missing | 3 | Drop: no target, cannot train or evaluate. |
| Subvillage heads (`RTHEAD == 1`) | 640 | Drop: added on purpose, not sampled, and better off (27% vs 36% poor). |
| `se2_139` coded 1/3 instead of 0/1 | all | Recode 3 → 0. |
| Head education missing (all 4 `hheduc` = 0) | 8 | Otherwise silently read as "no school" (`hheduc1`, the reference). Add an indicator `hheduc_missing`, then drop `hheduc1`. |
| `depratio` missing | 228 | All are households with no member aged 15–64 (division by zero). Add `no_workingage`; filling happens in the pipeline. |
| `pcfloor` missing | 52 | Add `pcfloor_missing`; median filled in the pipeline (training folds only). |
| A few missing 0/1 dummies (`toilet` 8, `se2_122_123` 4, `water`, `se5_110`, …) | ≤ 8 each | Leave missing; most frequent value filled in the pipeline. |
| `hr06xx` count missing because the item is not owned (dummy = 0) | most rows | Fill with 0: "not owned" means 0 items, by definition (no learning from data, so no leakage). |
| `hr06xx` count missing although the dummy says owned | 61 jewelry, 1 each motorcycle, car, chicken, pig | Leave missing (0 would contradict the dummy); filled in the pipeline. |
| `pcexppred` < 0 | 149 | Keep. `PMTSCORE = log(pcexppred)`, so these are exactly the 149 missing `PMTSCORE`. A negative prediction still ranks the household as poorest, which is what the baseline needs. |
| Households exactly on the poverty line | 6 (2 coded poor, 4 not) | Use `poor` as given; do not rebuild it with `<`. |

**Unusual but plausible: keep (lecture 106: outlier treatment depends on the task)**
- `hhage` of 14 and 105 (2 households): possible for a household head; no reason to treat as an error.
- `pcfloor` > 200 m² per person (22 households): mostly single-person households with a large house (e.g. 250 m², 1 person). No outliers on the log scale (Tukey k = 3); Yeo–Johnson in the pipeline handles the skew.
- 2 Tukey outliers (k = 3) on log consumption: real households; the log target and the ranking-based metric already limit their influence. Removing target outliers would be cherry-picking.

</details>

## Column selection: which columns we keep and why

<details>
<summary><b>Click to open: how we decided which columns to keep or drop</b></summary>

The raw files have 347 (`main`) and 95 (`labels`) columns. We reduce them **before merging** (same order as Q2 → Q5 of the Big Assignment solution). The code lists what we **keep**, not what we drop: any column not listed is removed, so a leakage column cannot slip through because someone forgot it in a drop list. If we later need a dropped column, we add it to the keep list and rerun the notebook.

### Questions asked of every column, in order (the first "yes" decides)

**1. Is it consumption or derived from it? → drop (direct leakage).**
It contains the answer we want to predict. Consumption is never an input: it is too costly to measure for every household, which is the whole reason the government uses a proxy-means test.
- Dropped: `hhconsumption`, `wfood`, `mfood`, `mnonfood`, `CONSUMPTION`, `RANKCONSUMPTION`, `rank_pcexp`, `nhhrank_pcexp`, `lnpcexpe`, `epcexpe`, `avg_CONSUMPTION`, `povertygap`, `verypoor`, `nearpoor`, `middle`, `rich`, `poorhc`, `verypoorhc`, `middlehc`.
- Exception: `LOGCONSUMPTION` and `poor` are kept, because they are our **targets** (never features).

**2. Is it only known after the cutoff? → drop (leakage from the future).**
Our cutoff is the door-to-door asset survey, *before* allocation. Anything describing who was selected, how the experiment was run or how the targeting turned out is the *result* of the selection, and it mixes the experiment's three targeting methods (which would move us towards a causal question we don't ask).
- Dropped: `RTS`, `RTS_meeting`, `ranking*`, `nhhrank*`, `TREATRANK`, `MISTARGET*`, `maintreatment` and all `*subtreatment`, `PMT`, `COMMUNITY`, `HYBRID`, `ELITE*`, `meeting_order`, `facilitator_code`, `fund_order`, `village_order`, `quota*`.
- Exception: `share`, the village quota fixed **before** the experiment, is kept: the decision rule needs it (top `round(8 × share)` households per village). It is never a feature.

**3. Is it the government's score or part of it? → drop, except one baseline column.**
The model must not learn from the government's prediction or its pieces; otherwise "beating the formula" would partly be copying it.
- Dropped: `PMTSCORE`, `PMTRANK`, `rank_pmt`, `nhhrank_pmt`, all `c_*` (formula coefficients), `comp_*`, `comp2_*`, `*_avg`, `*_dm`, `temp*`.
- Exception: `pcexppred` is kept **only as the government-formula baseline**, never as a feature. (`PMTSCORE` is not used: it is missing for 149 households, and `pcexppred` gives the same ranking.)

**4. Is it self-reported or a social item? → drop, unless it is in the government formula.**
In a real programme survey people know their answers can decide who gets money, so self-reports are easy to bend. Social items (connections, leaders, neighbours' views) are not something an enumerator observes at the door.
- Dropped: `healthcare` ("can afford healthcare"), `shock`, `ETHICMINORITY`, `RELIGIOUSMINORITY`, `elite_connectedness`, `FAMILY*`, `say_*`, `*LEADER`.
- Exception: `credit` (ever received a loan) is self-reported **but kept**, because it is in the government formula. Our rule for set A is "the same information as the government"; dropping it would make the comparison unfair.

**5. Is it in the government formula? → keep as set A (main feature set).**
The raw file contains the government's coefficients (`c_*` columns), so we read the formula variables directly from the data: **62 variables** (dwelling, household, head, household education, credit, urban, village census, assets).
- Note: `se5_15_16` (goat/sheep) and `se2_138` (agricultural land) have coefficients in the formula, so they belong to **set A**, not set B as first written in our outline.

**6. Can an enumerator see or verify it at the door? → keep as set B (extension).**
Extra assets (rice cooker, mixer, fan, telephone, sewing machine, water pump, rice husker, boats), livestock (pigs, horses), non-agricultural land, floor quality (`FLOORTOPQUALITY`) and the `hr06xx` asset counts.
- `se2_110` (fan), `se5_13` (horse) and `se2_139` (non-agricultural land) have a `c_*` column, but it is empty: they are **not** in the formula, so they are set B.

**7. Needed to identify, merge, filter, split or apply the quota? → keep, with a stated reason.**

| Column | Why we keep it |
|---|---|
| `hhid`, `hhid_suseti` | Merge key (store as text: leading zeros, e.g. `"001025"`) |
| `hhea` | Village: quota groups, train/test split, CV groups |
| `kabu` | District: district dummies; test villages are drawn within each district |
| `prop` | Province: balance check, province hold-out |
| `share` | Village quota for the decision rule |
| `pcexppred` | Government-formula baseline |
| `LOGCONSUMPTION`, `poor` | Targets |
| `povline_poor` | Only to document/check `poor` |
| `RTHEAD` | Only to drop the subvillage heads, then removed |
| `age1564` | Only to build `no_workingage`, then removed |

`kab` is dropped because `kabu` already contains it.

**8. Anything else → drop.**

### Village-level variables stay in the clean data
The 14 inputs that are the same for every household in a village (11 village facilities, province, district, `urban`) cannot change the ranking *within* a village. Following the feedback, the **linear models** will not use them, but **boosting and the neural net** keep them (they can combine them with household features). So they stay in the clean file; each model chooses its columns later in `src/features.py`, not here in the cleaning.

### Open points to discuss in the group
- Agree that `se5_15_16` and `se2_138` move to set A (update the outline).
- Agree to keep `credit` despite being self-reported (in the formula).
- Do we need `povline_poor` in the clean file, or only for the check here?

</details>